# 01 — Five-source GEE extraction (Kanha Tiger Reserve)

**300 patches** → local `data/raw_tiffs/` (no Google Drive).

| # | Source | Exported |
|---|---|---|
| 1 | Sentinel-2 | RGB, optical stack, indices |
| 2 | Sentinel-1 | VV, VH |
| 3 | CHIRPS | Rainfall |
| 4 | ERA5-Land | Temp + soil moisture anomalies |
| 5 | NASADEM | Elevation, slope, aspect |

**Dates:** T0 = Jan–Feb 2022 | T1 = Jan–Feb 2023

**Run order:** Cells 1–4 (setup) → **Batch 1** → **Batch 2** → **Batch 3** → scalars/inventory.

Each patch = **15 GeoTIFFs** in `data/raw_tiffs/`. Skips files that already exist (safe to re-run).


## 1. Authenticate Earth Engine


In [1]:
from pathlib import Path
import json
import sys

import ee
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.gee_utils import (
    CHIRPS_SCALE,
    DEM_SCALE,
    ERA5_SCALE,
    S2_INDICES,
    S2_OPTICAL_BANDS,
    get_chirps_rainfall,
    get_dem_terrain,
    get_dominant_orbit_pass,
    get_era5_anomalies,
    get_s1_composite,
    get_s2_composite,
    masked_fraction,
    patch_geometry,
    reduce_coarse,
    reduce_fine,
    rgb_reflectance,
)

RAW_TIFFS = ROOT / "data" / "raw_tiffs"
RAW_META = ROOT / "data" / "raw_metadata"
RAW_TIFFS.mkdir(parents=True, exist_ok=True)

GCP_PROJECT_ID = "chintaphaniramavaibhav1"

try:
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine already initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine authenticated and initialized.")
print("ROOT:", ROOT)


Earth Engine already initialized.
ROOT: /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset


## 2. Kanha ROI, matched seasonal windows, patch size


In [2]:
roi_bbox = [80.433, 22.117, 81.050, 22.450]
roi = ee.Geometry.Rectangle(roi_bbox)

BEFORE_START, BEFORE_END = "2022-01-01", "2022-02-28"
AFTER_START, AFTER_END = "2023-01-01", "2023-02-28"
CLIM_START, CLIM_END = "2015-01-01", "2021-12-31"

PATCH_PIXELS = 64
PIXEL_SIZE_M = 10
PATCH_SIZE_M = PATCH_PIXELS * PIXEL_SIZE_M
MAX_SCENE_CLOUD_PCT = 20

# ── 300-patch scale (3 batches × 100) ───────────────────────────
MAX_PATCHES = 300
PATCH_BATCH_SIZE = 100
FILES_PER_PATCH = 15

print("ROI area (sq km):", roi.area().divide(1e6).getInfo())
print(f"T0: {BEFORE_START} -> {BEFORE_END}")
print(f"T1: {AFTER_START} -> {AFTER_END}")
print(f"Patches: {MAX_PATCHES} in batches of {PATCH_BATCH_SIZE}")


ROI area (sq km): 2350.680237371923
T0: 2022-01-01 -> 2022-02-28
T1: 2023-01-01 -> 2023-02-28
Patches: 300 in batches of 100


## 3. Build all five source composites


In [3]:
s2_t0, n_s2_t0 = get_s2_composite(roi, BEFORE_START, BEFORE_END, MAX_SCENE_CLOUD_PCT)
s2_t1, n_s2_t1 = get_s2_composite(roi, AFTER_START, AFTER_END, MAX_SCENE_CLOUD_PCT)

S1_ORBIT_PASS = get_dominant_orbit_pass(roi, BEFORE_START, AFTER_END)
s1_t0, n_s1_t0 = get_s1_composite(roi, BEFORE_START, BEFORE_END, S1_ORBIT_PASS)
s1_t1, n_s1_t1 = get_s1_composite(roi, AFTER_START, AFTER_END, S1_ORBIT_PASS)

chirps_t0, n_ch_t0 = get_chirps_rainfall(roi, BEFORE_START, BEFORE_END)
chirps_t1, n_ch_t1 = get_chirps_rainfall(roi, AFTER_START, AFTER_END)

era5_temp_t0, era5_sm_t0 = get_era5_anomalies(roi, BEFORE_START, BEFORE_END, CLIM_START, CLIM_END)
era5_temp_t1, era5_sm_t1 = get_era5_anomalies(roi, AFTER_START, AFTER_END, CLIM_START, CLIM_END)
era5_t0 = era5_temp_t0.addBands(era5_sm_t0)
era5_t1 = era5_temp_t1.addBands(era5_sm_t1)

dem = get_dem_terrain(roi)

t0_rgb = rgb_reflectance(s2_t0)
t1_rgb = rgb_reflectance(s2_t1)
delta_ndvi = s2_t1.select("NDVI").subtract(s2_t0.select("NDVI")).rename("delta_ndvi")
delta_nbr = s2_t1.select("NBR").subtract(s2_t0.select("NBR")).rename("delta_nbr")

print("S2 scenes T0/T1:", n_s2_t0.getInfo(), n_s2_t1.getInfo())
print("S1 orbit:", S1_ORBIT_PASS, "scenes T0/T1:", n_s1_t0.getInfo(), n_s1_t1.getInfo())
print("CHIRPS days T0/T1:", n_ch_t0.getInfo(), n_ch_t1.getInfo())
print("Five-source composites ready.")


S2 scenes T0/T1: 20 23
S1 orbit: DESCENDING scenes T0/T1: 15 20
CHIRPS days T0/T1: 58 58
Five-source composites ready.


## 4. Load 300 selected patches


In [4]:
csv_path = RAW_META / "selected_patches.csv"
if not csv_path.exists():
    raise FileNotFoundError(csv_path)

selected = pd.read_csv(csv_path).head(MAX_PATCHES)
print(selected["candidate_class"].value_counts())
print("patches:", len(selected))


candidate_class
stable      100
decline     100
increase    100
Name: count, dtype: int64
patches: 300


## 5. Local export helpers


In [5]:
import geemap
import time
from datetime import datetime

LOG_PATH = RAW_META / "local_export_log.txt"
STATE_PATH = RAW_META / "local_export_state.json"
EXPORTS_PER_PATCH = FILES_PER_PATCH


def _log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with LOG_PATH.open("a") as f:
        f.write(line + "\n")


def _load_state():
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text())
    return {"batches_done": []}


def _save_state(state):
    STATE_PATH.write_text(json.dumps(state, indent=2))


def export_specs(pid, geom):
    return [
        (t0_rgb.clip(geom), RAW_TIFFS / f"{pid}_T0_rgb.tif", PIXEL_SIZE_M),
        (t1_rgb.clip(geom), RAW_TIFFS / f"{pid}_T1_rgb.tif", PIXEL_SIZE_M),
        (s2_t0.select(S2_OPTICAL_BANDS).clip(geom), RAW_TIFFS / f"{pid}_T0_s2_optical.tif", PIXEL_SIZE_M),
        (s2_t1.select(S2_OPTICAL_BANDS).clip(geom), RAW_TIFFS / f"{pid}_T1_s2_optical.tif", PIXEL_SIZE_M),
        (s2_t0.select(S2_INDICES).clip(geom), RAW_TIFFS / f"{pid}_T0_indices.tif", PIXEL_SIZE_M),
        (s2_t1.select(S2_INDICES).clip(geom), RAW_TIFFS / f"{pid}_T1_indices.tif", PIXEL_SIZE_M),
        (s1_t0.select(["VV", "VH"]).clip(geom), RAW_TIFFS / f"{pid}_T0_s1.tif", PIXEL_SIZE_M),
        (s1_t1.select(["VV", "VH"]).clip(geom), RAW_TIFFS / f"{pid}_T1_s1.tif", PIXEL_SIZE_M),
        (chirps_t0.clip(geom), RAW_TIFFS / f"{pid}_T0_rainfall.tif", CHIRPS_SCALE),
        (chirps_t1.clip(geom), RAW_TIFFS / f"{pid}_T1_rainfall.tif", CHIRPS_SCALE),
        (era5_t0.clip(geom), RAW_TIFFS / f"{pid}_T0_era5.tif", ERA5_SCALE),
        (era5_t1.clip(geom), RAW_TIFFS / f"{pid}_T1_era5.tif", ERA5_SCALE),
        (dem.clip(geom), RAW_TIFFS / f"{pid}_dem.tif", DEM_SCALE),
        (delta_ndvi.clip(geom), RAW_TIFFS / f"{pid}_delta_ndvi.tif", PIXEL_SIZE_M),
        (delta_nbr.clip(geom), RAW_TIFFS / f"{pid}_delta_nbr.tif", PIXEL_SIZE_M),
    ]


def patch_file_count(pid):
    return sum(1 for _ in RAW_TIFFS.glob(f"{pid}_*.tif"))


def export_patch_local(pid, geom):
    wrote, skipped = 0, 0
    for image, out_path, scale in export_specs(pid, geom):
        if out_path.exists():
            skipped += 1
            continue
        geemap.ee_export_image(
            image,
            filename=str(out_path),
            scale=scale,
            region=geom,
            file_per_band=False,
        )
        wrote += 1
        print(f"  wrote {out_path.name}")
    return wrote, skipped


def export_batch(batch_start: int):
    batch_end = min(batch_start + PATCH_BATCH_SIZE, len(selected))
    batch_key = f"{batch_start}:{batch_end}"
    state = _load_state()
    if batch_key in state.get("batches_done", []):
        _log(f"Batch {batch_key} already done — skipping")
        return

    batch = selected.iloc[batch_start:batch_end]
    _log("=" * 50)
    _log(f"BATCH {batch_key} — {len(batch)} patches -> {RAW_TIFFS}")
    _log("=" * 50)

    t0 = time.time()
    total_wrote, total_skip, errors = 0, 0, []

    for i, (_, row) in enumerate(batch.iterrows(), 1):
        pid = str(row["patch_id"])
        geom = patch_geometry(float(row["lon"]), float(row["lat"]), PATCH_SIZE_M, roi)
        n_have = patch_file_count(pid)
        if n_have >= EXPORTS_PER_PATCH:
            _log(f"[{i}/{len(batch)}] {pid} complete ({n_have}/{EXPORTS_PER_PATCH}) — skip")
            total_skip += EXPORTS_PER_PATCH
            continue
        _log(f"[{i}/{len(batch)}] {pid} ({n_have}/{EXPORTS_PER_PATCH} files)")
        try:
            w, s = export_patch_local(pid, geom)
            total_wrote += w
            total_skip += s
        except Exception as exc:
            errors.append((pid, str(exc)))
            _log(f"  ERROR {pid}: {exc}")

    elapsed = (time.time() - t0) / 60
    if not errors:
        state.setdefault("batches_done", []).append(batch_key)
        _save_state(state)
        _log(f"Batch {batch_key} DONE in {elapsed:.1f} min | wrote={total_wrote} skipped={total_skip}")
    else:
        _log(f"Batch {batch_key} finished with {len(errors)} errors — re-run to retry")
        for pid, err in errors[:5]:
            _log(f"  failed: {pid}: {err}")

    n_tifs = len(list(RAW_TIFFS.glob("*.tif")))
    _log(f"Total local tifs: {n_tifs} (expect {len(selected) * EXPORTS_PER_PATCH} when all done)")

_log("Export helpers ready.")


[2026-09-16 08:19:26] Export helpers ready.


### Batch 1 — patches 0–99 (run after setup cells 1–4)

Downloads **~1,500 files** (100 × 15) to `data/raw_tiffs/`. Safe to re-run.


In [6]:
# Batch 1 — patches 0-99
export_batch(0)


[2026-09-16 08:19:26] ==================================================
[2026-09-16 08:19:26] BATCH 0:100 — 100 patches -> /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs
[2026-09-16 08:19:26] ==================================================
[2026-09-16 08:19:26] [1/100] p04370 (0/15 files)
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04370_T0_rgb.tif
  wrote p04370_T0_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04370_T1_rgb.tif
  wrote p04370_T1_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04370_T0_s2_optical.tif
  wrote p04370_T0_s2_optical.tif
Generating URL ...
Please

### Batch 2 — patches 100–199

Run after Batch 1 finishes.


In [7]:
# Batch 2 — patches 100-199
export_batch(100)


[2026-09-16 09:20:22] ==================================================
[2026-09-16 09:20:22] BATCH 100:200 — 100 patches -> /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs
[2026-09-16 09:20:22] ==================================================
[2026-09-16 09:20:22] [1/100] p05107 (0/15 files)
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p05107_T0_rgb.tif
  wrote p05107_T0_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p05107_T1_rgb.tif
  wrote p05107_T1_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p05107_T0_s2_optical.tif
  wrote p05107_T0_s2_optical.tif
Generating URL ...
Plea

### Batch 3 — patches 200–299

Run after Batch 2 finishes.


In [8]:
# Batch 3 — patches 200-299
export_batch(200)


[2026-09-16 10:27:30] ==================================================
[2026-09-16 10:27:30] BATCH 200:300 — 100 patches -> /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs
[2026-09-16 10:27:30] ==================================================
[2026-09-16 10:27:30] [1/100] p03587 (0/15 files)
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p03587_T0_rgb.tif
  wrote p03587_T0_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p03587_T1_rgb.tif
  wrote p03587_T1_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p03587_T0_s2_optical.tif
  wrote p03587_T0_s2_optical.tif
Generating URL ...
Plea

In [12]:
# ═══════════════════════════════════════════════════════════════════
# ADD 100 PATCHES (300 → 400) — run once, then Batch 4 below
# ═══════════════════════════════════════════════════════════════════
import numpy as np
from pathlib import Path

ADD_PER_CLASS = 25          # 25 × 3 classes = 100 new patches
CLASSES = ["stable", "decline", "increase"]
MIN_PATCH_DISTANCE_M = PATCH_SIZE_M * 1.5
MAX_CLOUD_FRACTION = 0.15

PARENT_STATS = RAW_META / "composite_stats.csv"
if not PARENT_STATS.exists():
    PARENT_STATS = ROOT.parent / "data" / "raw_metadata" / "composite_stats.csv"

existing = pd.read_csv(RAW_META / "selected_patches.csv")
print(f"Current patches: {len(existing)}")

stats = pd.read_csv(PARENT_STATS)
stats = stats[~stats["patch_id"].isin(existing["patch_id"])].copy()

decline_q = stats["ndvi_change"].quantile(0.15)
increase_q = stats["ndvi_change"].quantile(0.85)

def assign_class(row):
    if row["masked_frac_before"] > MAX_CLOUD_FRACTION or row["masked_frac_after"] > MAX_CLOUD_FRACTION:
        return "uncertain"
    if row["ndvi_change"] <= decline_q:
        return "decline"
    if row["ndvi_change"] >= increase_q:
        return "increase"
    if abs(row["ndvi_change"]) < 0.05:
        return "stable"
    return "uncertain"

stats["candidate_class"] = stats.apply(assign_class, axis=1)

def geodesic_m(p1, p2):
    lon1, lat1 = p1
    lon2, lat2 = p2
    m_per_deg_lat = 111_320
    m_per_deg_lon = 111_320 * np.cos(np.radians((lat1 + lat2) / 2))
    dx = (lon2 - lon1) * m_per_deg_lon
    dy = (lat2 - lat1) * m_per_deg_lat
    return (dx ** 2 + dy ** 2) ** 0.5

kept_pts = list(zip(existing["lon"], existing["lat"]))
new_parts = []

for cls in CLASSES:
    subset = stats[stats["candidate_class"] == cls].copy()
    keep_idx = []
    for idx, row in subset.iterrows():
        pt = (row["lon"], row["lat"])
        if all(geodesic_m(pt, kp) >= MIN_PATCH_DISTANCE_M for kp in kept_pts):
            keep_idx.append(idx)
            kept_pts.append(pt)
    subset = subset.loc[keep_idx]
    n = min(ADD_PER_CLASS, len(subset))
    picked = subset.sample(n, random_state=2) if n > 0 else subset
    new_parts.append(picked)
    print(f"  {cls}: picked {len(picked)} (pool after distance filter: {len(subset)})")

new_patches = pd.concat(new_parts).reset_index(drop=True)
selected = pd.concat([existing, new_patches], ignore_index=True)
selected.to_csv(RAW_META / "selected_patches.csv", index=False)

print(f"\nTotal patches now: {len(selected)}")
print(selected["candidate_class"].value_counts())
print("Saved:", RAW_META / "selected_patches.csv")

Current patches: 300
  stable: picked 25 (pool after distance filter: 242)
  decline: picked 22 (pool after distance filter: 22)
  increase: picked 25 (pool after distance filter: 26)

Total patches now: 372
candidate_class
stable      125
increase    125
decline     122
Name: count, dtype: int64
Saved: /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_metadata/selected_patches.csv


In [13]:
# ═══════════════════════════════════════════════════════════════════
# BATCH 4 — patches 300-399 (100 new patches → 1,500 files)
# ═══════════════════════════════════════════════════════════════════
selected = pd.read_csv(RAW_META / "selected_patches.csv")
print(f"Exporting from {len(selected)} patches total")
export_batch(300)

Exporting from 372 patches total
[2026-09-16 11:38:45] ==================================================
[2026-09-16 11:38:45] BATCH 300:372 — 72 patches -> /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs
[2026-09-16 11:38:45] ==================================================
[2026-09-16 11:38:45] [1/72] p04508 (0/15 files)
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04508_T0_rgb.tif
  wrote p04508_T0_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04508_T1_rgb.tif
  wrote p04508_T1_rgb.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_tiffs/p04508_T0_s2_optical.tif
  wrote p04508_T0_s2_opti

## 6. Patch-level scalars (fixes the old ERA5/CHIRPS/DEM `None` bug)


In [14]:
def compute_patch_scalars(lon, lat):
    geom = patch_geometry(lon, lat, PATCH_SIZE_M, roi)

    def s2_mean(img, band):
        return img.select(band).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom, scale=PIXEL_SIZE_M, maxPixels=1e8
        ).get(band)

    keys = [
        "ndvi_t0", "ndvi_t1", "evi_t0", "evi_t1", "ndmi_t0", "ndmi_t1", "nbr_t0", "nbr_t1",
        "vv_t0", "vv_t1", "vh_t0", "vh_t1",
        "masked_frac_t0", "masked_frac_t1",
        "era5_temp_anomaly_t0", "era5_temp_anomaly_t1",
        "era5_sm_anomaly_t0", "era5_sm_anomaly_t1",
        "chirps_rainfall_t0", "chirps_rainfall_t1",
        "elevation_m", "slope_deg", "aspect_deg",
    ]
    values = ee.List([
        s2_mean(s2_t0, "NDVI"), s2_mean(s2_t1, "NDVI"),
        s2_mean(s2_t0, "EVI"), s2_mean(s2_t1, "EVI"),
        s2_mean(s2_t0, "NDMI"), s2_mean(s2_t1, "NDMI"),
        s2_mean(s2_t0, "NBR"), s2_mean(s2_t1, "NBR"),
        s2_mean(s1_t0, "VV"), s2_mean(s1_t1, "VV"),
        s2_mean(s1_t0, "VH"), s2_mean(s1_t1, "VH"),
        masked_fraction(s2_t0.clip(geom), geom),
        masked_fraction(s2_t1.clip(geom), geom),
        reduce_coarse(era5_temp_t0, lon, lat, "temp_anomaly", ERA5_SCALE),
        reduce_coarse(era5_temp_t1, lon, lat, "temp_anomaly", ERA5_SCALE),
        reduce_coarse(era5_sm_t0, lon, lat, "soil_moisture_anomaly", ERA5_SCALE),
        reduce_coarse(era5_sm_t1, lon, lat, "soil_moisture_anomaly", ERA5_SCALE),
        reduce_coarse(chirps_t0, lon, lat, "rainfall_mm", CHIRPS_SCALE),
        reduce_coarse(chirps_t1, lon, lat, "rainfall_mm", CHIRPS_SCALE),
        reduce_fine(dem, geom, "elevation_m", DEM_SCALE),
        reduce_fine(dem, geom, "slope_deg", DEM_SCALE),
        reduce_fine(dem, geom, "aspect_deg", DEM_SCALE),
    ]).getInfo()
    out = dict(zip(keys, values))
    out["s1_orbit_pass"] = S1_ORBIT_PASS
    out["s1_n_t0"] = n_s1_t0.getInfo()
    out["s1_n_t1"] = n_s1_t1.getInfo()
    out["t0_start"], out["t0_end"] = BEFORE_START, BEFORE_END
    out["t1_start"], out["t1_end"] = AFTER_START, AFTER_END
    return out


scalar_path = RAW_META / "patch_scalars.json"
scalars = {}
if scalar_path.exists():
    scalars = json.loads(scalar_path.read_text())

# Reuse parent CSV values when present so you are not blocked on extra EE getInfo calls.
csv_alias = {
    "ndvi_before": "ndvi_t0", "ndvi_after": "ndvi_t1",
    "evi_before": "evi_t0", "evi_after": "evi_t1",
    "ndmi_before": "ndmi_t0", "ndmi_after": "ndmi_t1",
    "nbr_before": "nbr_t0", "nbr_after": "nbr_t1",
    "vv_before": "vv_t0", "vv_after": "vv_t1",
    "vh_before": "vh_t0", "vh_after": "vh_t1",
    "masked_frac_before": "masked_frac_t0", "masked_frac_after": "masked_frac_t1",
    "era5_temp_anomaly_before": "era5_temp_anomaly_t0",
    "era5_temp_anomaly_after": "era5_temp_anomaly_t1",
    "era5_sm_anomaly_before": "era5_sm_anomaly_t0",
    "era5_sm_anomaly_after": "era5_sm_anomaly_t1",
    "chirps_rainfall_before": "chirps_rainfall_t0",
    "chirps_rainfall_after": "chirps_rainfall_t1",
    "s1_n_before": "s1_n_t0", "s1_n_after": "s1_n_t1",
}

for _, row in selected.iterrows():
    pid = str(row["patch_id"])
    rec = {new: row[old] for old, new in csv_alias.items() if old in row.index}
    for col in ("elevation_m", "slope_deg", "aspect_deg", "s1_orbit_pass", "fire_detected", "candidate_class"):
        if col in row.index:
            rec[col] = row[col]
    rec["t0_start"], rec["t0_end"] = BEFORE_START, BEFORE_END
    rec["t1_start"], rec["t1_end"] = AFTER_START, AFTER_END
    scalars[pid] = rec

scalar_path.write_text(json.dumps(scalars, indent=2, default=str))
print("wrote", scalar_path, "n=", len(scalars))
print("sample keys", sorted(next(iter(scalars.values())).keys())[:12], "...")


wrote /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/data/raw_metadata/patch_scalars.json n= 372
sample keys ['aspect_deg', 'candidate_class', 'chirps_rainfall_t0', 'chirps_rainfall_t1', 'elevation_m', 'era5_sm_anomaly_t0', 'era5_sm_anomaly_t1', 'era5_temp_anomaly_t0', 'era5_temp_anomaly_t1', 'evi_t0', 'evi_t1', 'fire_detected'] ...


## 7. Inventory


In [15]:
from collections import Counter
stems = [p.name.split("_", 1)[-1] if "_" in p.name else p.name for p in RAW_TIFFS.glob("*.tif")]
print("tif count", len(list(RAW_TIFFS.glob("*.tif"))))
print(Counter(stems).most_common(20))


tif count 5769
[('T1_rgb.tif', 385), ('T0_era5.tif', 385), ('T0_indices.tif', 385), ('delta_ndvi.tif', 385), ('T1_rainfall.tif', 385), ('T1_s1.tif', 385), ('T0_rainfall.tif', 385), ('T1_indices.tif', 385), ('dem.tif', 385), ('T0_s1.tif', 385), ('T1_era5.tif', 384), ('T0_s2_optical.tif', 384), ('T1_s2_optical.tif', 384), ('delta_nbr.tif', 384), ('T0_rgb.tif', 383)]
